# 18.07 - Video Reading

**Notebook type:** Solution notebook with complete implementations, smoke checks, and test cases.

**Daily output:** A robust `video_to_frames(video_path, n_frames=16)` function, tested on videos with several lengths.

Today turns a video file into a deterministic, fixed-length RGB frame array. The key is to distinguish container metadata from frames that can actually be decoded.

## Core Ideas

### 1. A video is an ordered sequence of images

OpenCV's `cv2.VideoCapture` exposes both metadata and decoded frames. Useful properties include:

- **FPS**: nominal frames per second.
- **Reported frame count**: the container's estimate of how many frames exist.
- **Width and height**: decoded frame size.
- **Duration estimate**: `reported_frame_count / fps` when FPS is valid.

Container metadata is useful but not guaranteed to be exact, especially for damaged or unusual files.

### 2. OpenCV decodes BGR, most ML pipelines expect RGB

A frame returned by OpenCV has shape `[H, W, 3]`, dtype `uint8`, and BGR channel order. Convert it with `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)` before visualization or model preprocessing.

### 3. Uniform temporal sampling

For `D` decodable frames and `T` requested samples, use rounded positions from `0` through `D - 1`:

```text
indices = round(linspace(0, D - 1, T))
```

This includes the beginning and end of the clip. If `D < T`, repeated indices are intentional: the result still has exactly `T` frames.

### 4. Missing or undecodable frames

A robust reference implementation decodes sequentially until `read()` fails, then samples from the frames that were actually decoded. This avoids trusting an incorrect header frame count. If no frame can be decoded, raise a clear error.

Sequential decoding is simple and reliable, but it stores all decoded frames. For long production videos, seek-based decoding or streaming/reservoir strategies can reduce memory use; those optimizations need extra care because seeking is codec-dependent.

### 5. Shape contract for downstream work

This notebook returns a contiguous NumPy array with shape `[T, H, W, C]`, where `T = n_frames` and `C = 3`. The next study day can convert it to a video tensor such as `[T, C, H, W]`.

In [ ]:
import os
import csv
import cv2
import numpy as np

SEED = 42
np.random.seed(SEED)

DEMO_DIR = "_day18_video_data"
DEFAULT_N_FRAMES = 16

## Prepared Video Data

The cell below creates three deterministic MJPG/AVI clips with 5, 16, and 29 frames. Their strong red/green/blue patterns make channel-order and temporal-change checks visible. It also writes `manifest.csv` beside the clips.

This is provided setup code, not a learner TODO.

**Return structure — `make_day18_demo_videos(output_dir)`:**

- Returns a `list[dict]` with exactly three items, one per generated video.
- Each dictionary has:
  - `video_path`: `str`, relative path to the AVI file.
  - `num_frames`: `int`, number of frames written.
  - `fps`: `float`, nominal frames per second.
  - `width`: `int`, frame width in pixels.
  - `height`: `int`, frame height in pixels.
- Side effects: creates `output_dir`, three `.avi` files, and `output_dir/manifest.csv`.

In [ ]:
def make_day18_demo_videos(output_dir=DEMO_DIR):
    os.makedirs(output_dir, exist_ok=True)

    fps = 8.0
    width, height = 64, 48
    specs = [
        ("short_05.avi", 5),
        ("exact_16.avi", 16),
        ("long_29.avi", 29),
    ]
    records = []
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")

    for filename, num_frames in specs:
        video_path = os.path.join(output_dir, filename)
        writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))
        if not writer.isOpened():
            raise RuntimeError("OpenCV could not create the demo video: " + video_path)

        for frame_index in range(num_frames):
            frame_bgr = np.zeros((height, width, 3), dtype=np.uint8)
            frame_bgr[:, :, 0] = np.clip(20 + frame_index * 3, 0, 255)
            frame_bgr[:, :, 1] = np.clip(
                45 + np.arange(width, dtype=np.uint8)[None, :] // 3 + frame_index * 2,
                0,
                255,
            )
            frame_bgr[:, :, 2] = np.clip(
                180 - frame_index * 2 + np.arange(height, dtype=np.uint8)[:, None] // 4,
                0,
                255,
            )
            cv2.putText(
                frame_bgr,
                str(frame_index),
                (4, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                1,
                cv2.LINE_AA,
            )
            writer.write(frame_bgr)

        writer.release()
        records.append(
            {
                "video_path": video_path,
                "num_frames": int(num_frames),
                "fps": float(fps),
                "width": int(width),
                "height": int(height),
            }
        )

    manifest_path = os.path.join(output_dir, "manifest.csv")
    with open(manifest_path, "w", newline="", encoding="utf-8") as manifest_file:
        writer = csv.DictWriter(
            manifest_file,
            fieldnames=["video_path", "num_frames", "fps", "width", "height"],
        )
        writer.writeheader()
        writer.writerows(records)

    return records


demo_videos = make_day18_demo_videos()
print("Prepared videos:")
for item in demo_videos:
    print(item)

## Exercise 18-A: Inspect Video Metadata

Implement `inspect_video(video_path)`. Open the clip, read its nominal metadata, release the capture, and return a structured summary. Reject missing files and captures that cannot be opened.

Remember: `reported_frame_count` is header metadata, not proof that every frame is decodable.

**Return structure — `inspect_video(video_path)`:**

- Returns one `dict` with exactly these keys:
  - `video_path`: `str`, the input path.
  - `fps`: `float`, nominal FPS (may be `0.0` for an unknown value).
  - `reported_frame_count`: `int`, non-negative header frame count.
  - `width`: `int`, non-negative frame width.
  - `height`: `int`, non-negative frame height.
  - `duration_seconds`: `float | None`; `reported_frame_count / fps` when FPS is positive, otherwise `None`.
- Raises `FileNotFoundError` for a missing path and `ValueError` if OpenCV cannot open it.

In [ ]:
def inspect_video(video_path):
    if not isinstance(video_path, str) or not video_path:
        raise TypeError("video_path must be a non-empty string")
    if not os.path.isfile(video_path):
        raise FileNotFoundError("Video file does not exist: " + video_path)

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        capture.release()
        raise ValueError("OpenCV could not open video: " + video_path)

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    reported_frame_count = max(0, int(capture.get(cv2.CAP_PROP_FRAME_COUNT)))
    width = max(0, int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)))
    height = max(0, int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    capture.release()

    duration_seconds = reported_frame_count / fps if fps > 0 else None
    return {
        "video_path": video_path,
        "fps": fps,
        "reported_frame_count": reported_frame_count,
        "width": width,
        "height": height,
        "duration_seconds": duration_seconds,
    }


# Smoke check: inspect one prepared clip immediately.
metadata_smoke = inspect_video(demo_videos[0]["video_path"])
print("Metadata smoke check:", metadata_smoke)

## Exercise 18-B: Build Uniform Sample Indices

Implement `uniform_frame_indices(frame_count, n_frames)`. It must always return exactly `n_frames` monotonically non-decreasing positions, including the first and last available frame. Repeated positions are expected when a clip is short.

**Return structure — `uniform_frame_indices(frame_count, n_frames)`:**

- Returns a `numpy.ndarray`.
- Shape: `[T]`, where `T = n_frames`.
- Dtype: `numpy.int64`.
- Values: integers in `[0, frame_count - 1]`, monotonically non-decreasing.
- Device: CPU (NumPy array).
- Raises `ValueError` when `frame_count <= 0` or `n_frames <= 0`.

In [ ]:
def uniform_frame_indices(frame_count, n_frames):
    if int(frame_count) <= 0:
        raise ValueError("frame_count must be positive")
    if int(n_frames) <= 0:
        raise ValueError("n_frames must be positive")

    positions = np.linspace(0, int(frame_count) - 1, num=int(n_frames))
    return np.rint(positions).astype(np.int64)


# Smoke check: a five-frame clip sampled into eight slots repeats positions.
indices_smoke = uniform_frame_indices(frame_count=5, n_frames=8)
print("Index smoke check:", indices_smoke)

## Exercise 18-C: Decode and Uniformly Sample RGB Frames

Implement the day's main output: `video_to_frames(video_path, n_frames=16)`.

Decode sequentially and retain only valid three-channel frames. Sample using the number of successfully decoded frames—not only the header count. Convert BGR to RGB before stacking. A short clip should still return exactly `n_frames` through repeated samples.

**Return structure — `video_to_frames(video_path, n_frames=16)`:**

- Returns a contiguous `numpy.ndarray`.
- Shape: `[T, H, W, 3]`, where `T = n_frames`, and `H/W` are decoded frame height/width.
- Dtype: `numpy.uint8`.
- Channel order: RGB.
- Device: CPU (NumPy array).
- Raises `FileNotFoundError` for a missing path, `ValueError` when `n_frames <= 0`, when the capture cannot open, or when zero valid frames decode.

In [ ]:
def video_to_frames(video_path, n_frames=16):
    if not isinstance(video_path, str) or not video_path:
        raise TypeError("video_path must be a non-empty string")
    if not os.path.isfile(video_path):
        raise FileNotFoundError("Video file does not exist: " + video_path)
    if int(n_frames) <= 0:
        raise ValueError("n_frames must be positive")

    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        capture.release()
        raise ValueError("OpenCV could not open video: " + video_path)

    decoded_rgb_frames = []
    while True:
        ok, frame_bgr = capture.read()
        if not ok:
            break
        if frame_bgr is None or frame_bgr.ndim != 3 or frame_bgr.shape[2] != 3:
            continue
        decoded_rgb_frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    capture.release()

    if not decoded_rgb_frames:
        raise ValueError("No valid frames could be decoded from: " + video_path)

    sample_indices = uniform_frame_indices(len(decoded_rgb_frames), int(n_frames))
    sampled_frames = np.stack(
        [decoded_rgb_frames[int(index)] for index in sample_indices],
        axis=0,
    )
    return np.ascontiguousarray(sampled_frames, dtype=np.uint8)


# Smoke check: decode the short clip into a fixed eight-frame RGB array.
frames_smoke = video_to_frames(demo_videos[0]["video_path"], n_frames=8)
print(
    "Frame smoke check:",
    frames_smoke.shape,
    frames_smoke.dtype,
    "RGB mean:",
    frames_smoke[0].mean(axis=(0, 1)),
)

## Exercise 18-D: Compare Several Video Lengths

Implement `compare_video_lengths(video_paths, n_frames)`. Use the earlier functions to create a compact report proving that short, exact-length, and long clips all become the same output shape.

The first and last RGB means are lightweight temporal/channel diagnostics; they are not a substitute for viewing frames.

**Return structure — `compare_video_lengths(video_paths, n_frames)`:**

- Returns a `list[dict]` with one item per input path, preserving input order.
- Each dictionary has exactly:
  - `video_path`: `str`.
  - `reported_frame_count`: `int`.
  - `fps`: `float`.
  - `sampled_shape`: `tuple[int, int, int, int]` equal to `(T, H, W, 3)`.
  - `sampled_dtype`: `str`, expected to be `"uint8"`.
  - `first_rgb_mean`: `list[float]` of length 3 in RGB order.
  - `last_rgb_mean`: `list[float]` of length 3 in RGB order.
- `T = n_frames`; arrays are summarized rather than returned in the report.

In [ ]:
def compare_video_lengths(video_paths, n_frames):
    report = []
    for video_path in video_paths:
        metadata = inspect_video(video_path)
        sampled = video_to_frames(video_path, n_frames=n_frames)
        report.append(
            {
                "video_path": video_path,
                "reported_frame_count": metadata["reported_frame_count"],
                "fps": metadata["fps"],
                "sampled_shape": tuple(int(value) for value in sampled.shape),
                "sampled_dtype": str(sampled.dtype),
                "first_rgb_mean": [
                    float(value) for value in sampled[0].mean(axis=(0, 1))
                ],
                "last_rgb_mean": [
                    float(value) for value in sampled[-1].mean(axis=(0, 1))
                ],
            }
        )
    return report


# Smoke check: compare all prepared video lengths immediately.
length_report_smoke = compare_video_lengths(
    [item["video_path"] for item in demo_videos],
    n_frames=DEFAULT_N_FRAMES,
)
for row in length_report_smoke:
    print("Length smoke check:", row)

## Missing Frames and Production Trade-offs

The main function samples from successfully decoded frames. If a file reports 100 frames but decoding stops after 93, the sampling range is `0..92`; the output remains fixed-length. This is safer than generating indices from 100 and assuming every seek succeeds.

For larger systems, record both reported and decoded counts in logs. Decide explicitly whether a completely unreadable video should raise, be skipped by a dataset, or produce a sentinel sample. This notebook raises because silently returning fabricated pixels would hide a data-quality problem.

## Test Cases

Run this cell after completing Exercises 18-A through 18-D. It checks fixture files, metadata, uniform index behavior, output shape/dtype/layout, RGB conversion, repeated samples for short clips, temporal change, multiple video lengths, and error handling.

A correct implementation prints exactly `Day 18 tests passed`.

**Return structure — `run_day18_tests()`:**

- Returns `None`.
- Success is communicated by completing all assertions and printing `Day 18 tests passed`.
- Failure is communicated by an `AssertionError` (or the expected implementation error surfacing).

In [ ]:
def run_day18_tests():
    required_names = [
        "inspect_video",
        "uniform_frame_indices",
        "video_to_frames",
        "compare_video_lengths",
    ]
    for name in required_names:
        assert name in globals(), "Missing required implementation: " + name

    assert len(demo_videos) == 3
    assert os.path.isfile(os.path.join(DEMO_DIR, "manifest.csv"))
    for fixture in demo_videos:
        assert os.path.isfile(fixture["video_path"])

    metadata = inspect_video(demo_videos[1]["video_path"])
    assert set(metadata) == {
        "video_path",
        "fps",
        "reported_frame_count",
        "width",
        "height",
        "duration_seconds",
    }
    assert metadata["reported_frame_count"] == 16
    assert abs(metadata["fps"] - 8.0) < 0.25
    assert (metadata["width"], metadata["height"]) == (64, 48)
    assert metadata["duration_seconds"] is not None
    assert abs(metadata["duration_seconds"] - 2.0) < 0.1

    short_indices = uniform_frame_indices(5, 8)
    assert short_indices.dtype == np.int64
    assert short_indices.shape == (8,)
    assert np.array_equal(short_indices, np.array([0, 1, 1, 2, 2, 3, 3, 4]))
    assert np.all(short_indices[:-1] <= short_indices[1:])

    one_index = uniform_frame_indices(1, 4)
    assert np.array_equal(one_index, np.zeros(4, dtype=np.int64))

    short_frames = video_to_frames(demo_videos[0]["video_path"], n_frames=8)
    assert short_frames.shape == (8, 48, 64, 3)
    assert short_frames.dtype == np.uint8
    assert short_frames.flags["C_CONTIGUOUS"]
    assert np.array_equal(short_frames[1], short_frames[2])
    assert np.array_equal(short_frames[4], short_frames[5]) is False
    assert float(short_frames[0, :, :, 0].mean()) > float(
        short_frames[0, :, :, 2].mean()
    ), "Frames should be RGB, with the generated red channel stronger than blue"

    long_frames = video_to_frames(demo_videos[2]["video_path"], n_frames=16)
    assert long_frames.shape == (16, 48, 64, 3)
    assert not np.array_equal(long_frames[0], long_frames[-1])

    report = compare_video_lengths(
        [item["video_path"] for item in demo_videos],
        n_frames=16,
    )
    assert len(report) == 3
    expected_keys = {
        "video_path",
        "reported_frame_count",
        "fps",
        "sampled_shape",
        "sampled_dtype",
        "first_rgb_mean",
        "last_rgb_mean",
    }
    for row in report:
        assert set(row) == expected_keys
        assert row["sampled_shape"] == (16, 48, 64, 3)
        assert row["sampled_dtype"] == "uint8"
        assert len(row["first_rgb_mean"]) == 3
        assert len(row["last_rgb_mean"]) == 3

    try:
        uniform_frame_indices(0, 4)
        raise AssertionError("frame_count=0 should raise ValueError")
    except ValueError:
        pass

    try:
        video_to_frames("missing_day18_video.avi", n_frames=4)
        raise AssertionError("A missing video should raise FileNotFoundError")
    except FileNotFoundError:
        pass

    print("Day 18 tests passed")


run_day18_tests()

## Day 18 Checklist

- [ ] I can explain why reported frame count may differ from decodable frame count.
- [ ] I read FPS, frame count, width, height, and duration metadata.
- [ ] I generated monotonic uniform sample indices and handled short clips.
- [ ] I decoded OpenCV BGR frames and returned RGB `uint8` frames.
- [ ] My `video_to_frames` output has fixed shape `[T, H, W, 3]`.
- [ ] I tested short, exact-length, and long videos.
- [ ] I understand the memory trade-off of decoding a full video before sampling.
- [ ] The final test cell prints `Day 18 tests passed`.